In [0]:
# ============================================================================
# Rearc Data Quest — Spark Declarative Pipeline
# ----------------------------------------------------------------------------
# Bronze / Silver / Gold pipeline for BLS productivity data + population API.
# Bronze: raw files from Volume, minimal transformation (trim whitespace).
# Silver: typed, deduplicated, enriched with lookup tables.
# Gold:   three analytical answer tables (SQL primary, PySpark alternatives).
# ============================================================================

import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

# Volume paths
BLS_VOLUME = "/Volumes/rearc_quest/bronze/raw_data/bls_pr"
POP_VOLUME = "/Volumes/rearc_quest/bronze/raw_data/population/population.json"

# Helper: read a BLS tab-delimited file and trim column names + values
def read_bls_file(filename):
    path = f"{BLS_VOLUME}/{filename}"
    df = spark.read.csv(path, sep="\t", header=True, inferSchema=False)
    # BLS pads column names with spaces (e.g., "series_id        ", "       value")
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip())
    # Trim all string columns
    for c in df.columns:
        df = df.withColumn(c, trim(col(c)))
    return df


In [0]:
# ============================================================================
# BRONZE LAYER — Raw ingestion from Volume files
# ============================================================================

# --- Bronze: BLS data (pr.data.0.Current) ---
@dlt.table(
    name="bronze.bls_data",
    table_properties={"quality": "bronze"},
    comment="Raw BLS productivity data from pr.data.0.Current"
)
@dlt.expect("series_id_not_null", "series_id IS NOT NULL")
@dlt.expect("year_not_null", "year IS NOT NULL")
def bronze_bls_data():
    return read_bls_file("pr.data.0.Current")

# --- Bronze: BLS series metadata (pr.series) ---
@dlt.table(
    name="bronze.bls_series",
    table_properties={"quality": "bronze"},
    comment="Raw BLS series metadata from pr.series"
)
@dlt.expect("series_id_not_null", "series_id IS NOT NULL")
def bronze_bls_series():
    return read_bls_file("pr.series")

# --- Bronze: BLS measure lookup (pr.measure) ---
@dlt.table(
    name="bronze.bls_measure",
    table_properties={"quality": "bronze"},
    comment="BLS measure lookup table"
)
@dlt.expect("measure_code_not_null", "measure_code IS NOT NULL")
def bronze_bls_measure():
    return read_bls_file("pr.measure")

# --- Bronze: BLS sector lookup (pr.sector) ---
@dlt.table(
    name="bronze.bls_sector",
    table_properties={"quality": "bronze"},
    comment="BLS sector lookup table"
)
@dlt.expect("sector_code_not_null", "sector_code IS NOT NULL")
def bronze_bls_sector():
    return read_bls_file("pr.sector")

# --- Bronze: BLS class lookup (pr.class) ---
@dlt.table(
    name="bronze.bls_class",
    table_properties={"quality": "bronze"},
    comment="BLS class lookup table"
)
@dlt.expect("class_code_not_null", "class_code IS NOT NULL")
def bronze_bls_class():
    return read_bls_file("pr.class")

# --- Bronze: BLS period lookup (pr.period) ---
@dlt.table(
    name="bronze.bls_period",
    table_properties={"quality": "bronze"},
    comment="BLS period lookup table"
)
@dlt.expect("period_not_null", "period IS NOT NULL")
def bronze_bls_period():
    return read_bls_file("pr.period")

# --- Bronze: BLS duration lookup (pr.duration) ---
@dlt.table(
    name="bronze.bls_duration",
    table_properties={"quality": "bronze"},
    comment="BLS duration lookup table"
)
@dlt.expect("duration_code_not_null", "duration_code IS NOT NULL")
def bronze_bls_duration():
    return read_bls_file("pr.duration")

# --- Bronze: BLS seasonal lookup (pr.seasonal) ---
@dlt.table(
    name="bronze.bls_seasonal",
    table_properties={"quality": "bronze"},
    comment="BLS seasonal lookup table"
)
@dlt.expect("seasonal_code_not_null", "Seasonal_code IS NOT NULL")
def bronze_bls_seasonal():
    return read_bls_file("pr.seasonal")

# --- Bronze: Population API JSON ---
@dlt.table(
    name="bronze.population",
    table_properties={"quality": "bronze"},
    comment="Raw population data from DataUSA API (ACS 1-year estimates)"
)
@dlt.expect("year_not_null", "Year IS NOT NULL")
@dlt.expect("population_not_null", "Population IS NOT NULL")
def bronze_population():
    # Read the multi-line JSON file and explode the 'data' array
    raw_df = spark.read.option("multiLine", "true").json(POP_VOLUME)
    # Explode the data array into individual records
    exploded = raw_df.select(explode(col("data")).alias("record"))
    return exploded.select(
        col("record.Year").alias("Year"),
        col("record.Nation").alias("Nation"),
        col("record.`Nation ID`").alias("Nation_ID"),
        col("record.Population").alias("Population")
    )


In [0]:
# ============================================================================
# SILVER LAYER — Typed, deduplicated, enriched
# ============================================================================

# --- Silver: BLS data (typed + deduplicated) ---
@dlt.table(
    name="silver.bls_data",
    table_properties={"quality": "silver"},
    comment="Cleaned BLS data: typed columns, deduplicated on (series_id, year, period)"
)
@dlt.expect("series_id_not_null", "series_id IS NOT NULL")
@dlt.expect("year_not_null", "year IS NOT NULL")
@dlt.expect("value_not_null", "value IS NOT NULL")
@dlt.expect("year_valid", "year > 1900")
def silver_bls_data():
    return (
        dlt.read("bronze.bls_data")
        .withColumn("year", col("year").cast("int"))
        .withColumn("value", col("value").cast("double"))
        .dropDuplicates(["series_id", "year", "period"])
    )

# --- Silver: BLS series enriched (with human-readable label) ---
@dlt.table(
    name="silver.bls_series_enriched",
    table_properties={"quality": "silver"},
    comment="BLS series metadata enriched with lookup tables for human-readable labels"
)
@dlt.expect("series_id_not_null", "series_id IS NOT NULL")
@dlt.expect("sector_code_not_null", "sector_code IS NOT NULL")
@dlt.expect("measure_code_not_null", "measure_code IS NOT NULL")
def silver_bls_series_enriched():
    series_df = dlt.read("bronze.bls_series")
    measure_df = dlt.read("bronze.bls_measure")
    sector_df = dlt.read("bronze.bls_sector")
    class_df = dlt.read("bronze.bls_class")
    duration_df = dlt.read("bronze.bls_duration")
    seasonal_df = dlt.read("bronze.bls_seasonal")

    enriched = (
        series_df
        .join(sector_df, on="sector_code", how="left")
        .join(class_df, on="class_code", how="left")
        .join(measure_df, on="measure_code", how="left")
        .join(duration_df, on="duration_code", how="left")
        .join(seasonal_df, series_df["seasonal"] == seasonal_df["Seasonal_code"], how="left")
    )

    # Build human-readable label: "Sector - Class - Measure (Duration, Seasonal)"
    # e.g., "Manufacturing - All workers - Labor productivity (output per hour) (% Change same quarter 1 year ago, Seasonal Adjusted)"
    enriched = enriched.withColumn(
        "series_label",
        concat_ws(" - ",
            col("sector_name"),
            col("class_text")
        )
    )
    enriched = enriched.withColumn(
        "series_label",
        concat_ws(" ",
            col("series_label"),
            concat_ws(", ",
                col("measure_text"),
                col("duration_text")
            )
        )
    )

    # Select relevant columns
    return enriched.select(
        "series_id",
        "sector_code",
        "sector_name",
        "class_code",
        "class_text",
        "measure_code",
        "measure_text",
        "duration_code",
        "duration_text",
        "seasonal",
        "Seasonal_text",
        "base_year",
        "begin_year",
        "begin_period",
        "end_year",
        "end_period",
        "series_label"
    )

# --- Silver: Population (typed + deduplicated) ---
@dlt.table(
    name="silver.population",
    table_properties={"quality": "silver"},
    comment="Cleaned population data: typed columns, deduplicated on Year"
)
@dlt.expect("year_not_null", "Year IS NOT NULL")
@dlt.expect("population_not_null", "Population IS NOT NULL")
def silver_population():
    return (
        dlt.read("bronze.population")
        .withColumn("Year", col("Year").cast("int"))
        .withColumn("Population", col("Population").cast("long"))
        .dropDuplicates(["Year"])
    )


In [0]:
# ============================================================================
# GOLD LAYER — Analytical Answers
# ----------------------------------------------------------------------------
# Each Gold table is implemented in Spark SQL (primary) via spark.sql(),
# with a PySpark DataFrame API alternative included as commented code.
# Both implementations produce identical results; SQL is primary.
# ============================================================================

# === Q1: Population Statistics (2013-2018) ===================================
# What are the mean and standard deviation of the annual US population
# across 2013-2018 inclusive?
#
# PRIMARY: Spark SQL implementation

@dlt.table(
    name="gold.population_stats",
    table_properties={"quality": "gold"},
    comment="Q1: Mean and standard deviation of US population, 2013-2018"
)
def gold_population_stats():
    return spark.sql("""
        SELECT
            AVG(Population) AS mean_population,
            STDDEV(Population) AS stddev_population
        FROM silver.population
        WHERE Year BETWEEN 2013 AND 2018
    """)

# --- PySpark DataFrame API Alternative (Q1) ----------------------------------
# # @dlt.table(name="gold.population_stats_pyspark")
# # def gold_population_stats_pyspark():
# #     from pyspark.sql.functions import avg, stddev, col
# #     return (
# #         dlt.read("silver.population")
# #         .filter(col("Year").between(2013, 2018))
# #         .agg(
# #             avg("Population").alias("mean_population"),
# #             stddev("Population").alias("stddev_population")
# #         )
# #     )


In [0]:
# === Q2: Best Year Per Series ==============================================
# For every series_id, find the year with the largest sum of value across
# all its quarters. Include a human-readable series label.
#
# PRIMARY: Spark SQL implementation

@dlt.table(
    name="gold.best_year_per_series",
    table_properties={"quality": "gold"},
    comment="Q2: Best year (max sum of value) for each BLS series with human-readable label"
)
def gold_best_year_per_series():
    return spark.sql("""
        WITH yearly_sums AS (
            SELECT
                series_id,
                year,
                SUM(value) AS total_value
            FROM silver.bls_data
            GROUP BY series_id, year
        ),
        ranked AS (
            SELECT
                series_id,
                year,
                total_value,
                ROW_NUMBER() OVER (PARTITION BY series_id ORDER BY total_value DESC) AS rn
            FROM yearly_sums
        )
        SELECT
            r.series_id,
            r.year AS best_year,
            r.total_value,
            se.series_label
        FROM ranked r
        LEFT JOIN silver.bls_series_enriched se ON r.series_id = se.series_id
        WHERE r.rn = 1
    """)

# --- PySpark DataFrame API Alternative (Q2) ----------------------------------
# # from pyspark.sql.window import Window
# # from pyspark.sql.functions import sum as _sum, col, row_number
# #
# # @dlt.table(name="gold.best_year_per_series_pyspark")
# # def gold_best_year_per_series_pyspark():
# #     data_df = dlt.read("silver.bls_data")
# #     series_df = dlt.read("silver.bls_series_enriched")
# #
# #     yearly_sums = data_df.groupBy("series_id", "year") \
# #         .agg(_sum("value").alias("total_value"))
# #
# #     w = Window.partitionBy("series_id").orderBy(col("total_value").desc())
# #     ranked = yearly_sums.withColumn("rn", row_number().over(w))
# #
# #     best = ranked.filter(col("rn") == 1).drop("rn")
# #
# #     return best.join(series_df.select("series_id", "series_label"),
# #                      on="series_id", how="left")


In [0]:
# === Q3: Series Q01 Values Joined with Population ==========================
# For series_id = PRS30006032 and period = Q01, what was the value each year,
# joined with that year's population where available?
#
# PRIMARY: Spark SQL implementation

@dlt.table(
    name="gold.series_q01_population",
    table_properties={"quality": "gold"},
    comment="Q3: PRS30006032 Q01 values each year, joined with population"
)
def gold_series_q01_population():
    return spark.sql("""
        SELECT
            d.year,
            d.value,
            p.Population AS population
        FROM silver.bls_data d
        LEFT JOIN silver.population p ON d.year = p.Year
        WHERE d.series_id = 'PRS30006032'
            AND d.period = 'Q01'
        ORDER BY d.year
    """)

# --- PySpark DataFrame API Alternative (Q3) ----------------------------------
# # @dlt.table(name="gold.series_q01_population_pyspark")
# # def gold_series_q01_population_pyspark():
# #     data_df = dlt.read("silver.bls_data")
# #     pop_df = dlt.read("silver.population")
# #
# #     filtered = data_df \
# #         .filter((col("series_id") == "PRS30006032") & (col("period") == "Q01")) \
# #         .select("year", "value")
# #
# #     return filtered.join(
# #         pop_df.select(col("Year").alias("pop_year"), col("Population")),
# #         col("year") == col("pop_year"),
# #         "left"
# #     ).drop("pop_year").orderBy("year")
